##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multimodal Live API - Translation Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_LiveTranslate.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

**Preview**: The Live API is in preview.

This notebook demonstrates usage of the Gemini Live API for real-time audio translation. For an overview of new capabilities refer to the [Gemini Live API docs](https://ai.google.dev/gemini-api/docs/live-api/capabilities).

Some features of the API (such as low-latency bidirectional voice and video streaming using the local microphone and camera) are not supported in a standard Colab environment due to its headless cloud VM nature. To try full local hardware streaming, check out the CLI examples in the [Cookbook repository](https://github.com/google-gemini/cookbook/tree/main/quickstarts).

In this notebook, you will learn how to **stream and translate audio from a URL** in real-time using the Live Translation API, displaying live transcripts and playing the translated audio output.

## Setup

### Install SDK and Dependencies

The new **[Google Gen AI SDK](https://ai.google.dev/gemini-api/docs/sdks)** provides programmatic access to Gemini.

> **Note**: This notebook also uses `ffmpeg` to process the audio stream. `ffmpeg` is pre-installed in Google Colab environments.

In [ ]:
%pip install -U -q google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.5/832.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.8.0 which is incompatible.


### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for details.

In [ ]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

The client will pick up your API key from the environment variable.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

### Select a model

The Live Translation API uses the translation-capable Live models.

In [ ]:
MODEL = "gemini-3.5-live-translate-preview"  # @param ["gemini-3.5-live-translate-preview"] {"allow-input": true, "isTemplate": true}

### Import Modules

Import the necessary packages for handling async events, audio format streams, and file writing.

In [ ]:
import array
import asyncio
import contextlib
import wave

from IPython.display import display, Audio

from google import genai
from google.genai import types

### Helper to Write WAV Files

Let's define a helper context manager to write received audio chunks to a `.wav` file for playback in the notebook:

In [ ]:
@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

## Audio URL Streaming & Translation

You can stream audio in real-time to the Live API, and receive translated audio back. Here, you will stream audio from a public audio URL in 100ms chunks to mimic real-time audio input, and stream translation responses back.

### Helper for Streaming Audio URL

You define a helper function to stream audio from an HTTP URL and use `ffmpeg` to transcode it to raw PCM 16kHz mono audio.

In [ ]:
async def stream_audio_url(url: str, audio_queue: asyncio.Queue, sample_rate: int = 16000, channels: int = 1, chunk_size: int = 1600):
    """Streams audio from an HTTP URL, decoding it via ffmpeg and putting raw PCM bytes into the audio_queue."""
    print(f"\n[Info] Starting audio stream via ffmpeg from: {url}")
    # Spawn ffmpeg to decode stream to raw PCM 16kHz mono 16-bit
    process = await asyncio.create_subprocess_exec(
        'ffmpeg',
        '-i', url,
        '-f', 's16le',
        '-acodec', 'pcm_s16le',
        '-ar', str(sample_rate),
        '-ac', str(channels),
        '-',
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.DEVNULL
    )

    # 1600 samples * 2 bytes/sample (16-bit) = 3200 bytes per chunk
    chunk_size_bytes = chunk_size * 2
    bytes_per_second = sample_rate * 2
    start_time = asyncio.get_event_loop().time()
    bytes_sent = 0

    try:
        while True:
            data = await process.stdout.read(chunk_size_bytes)
            if not data:
                break

            await audio_queue.put(data)
            bytes_sent += len(data)

            # Rate limit to real-time speed (1.0x) so we simulate real mic streaming
            expected_elapsed = bytes_sent / bytes_per_second
            actual_elapsed = asyncio.get_event_loop().time() - start_time
            sleep_time = expected_elapsed - actual_elapsed
            if sleep_time > 0:
                await asyncio.sleep(sleep_time)
    except asyncio.CancelledError:
        pass
    finally:
        if process.returncode is None:
            try:
                process.terminate()
                await process.wait()
            except Exception:
                pass
        print("\n[Info] Audio stream finished.")

### Helper for Sending Audio & Receiving Translated Responses

Next, you define functions to push the audio chunks from the queue to the Live session, and to receive and print the source and translation transcripts.

In [ ]:
async def send_realtime(session, audio_queue: asyncio.Queue, sample_rate: int = 16000):
    """Sends audio from the input queue to the GenAI session."""
    try:
        while True:
            chunk = await audio_queue.get()
            await session.send_realtime_input(
                audio=types.Blob(
                    data=chunk,
                    mime_type=f"audio/pcm;rate={sample_rate}"
                )
            )
            audio_queue.task_done()
    except asyncio.CancelledError:
        pass

### Run Audio URL Translation

Now, set up the main translation runner. You will use a static audio URL: `https://storage.googleapis.com/generativeai-downloads/gemini-cookbook/audio/gemini-live-translate-sample.wav`.
The code runs the input audio stream, upload stream, and receiver concurrently in a `TaskGroup`. The translated Spanish audio is written to a wave file and played back.

## Next steps

This tutorial shows basic audio translation capabilities using the Multimodal Live API.

- Try it out in [Google AI Studio](https://aistudio.google.com/live?model=gemini-3.5-live-translate-preview)
- Read the [docs](https://ai.google.dev/gemini-api/docs/live-api/live-translate)
- Clone the [Live API examples from GitHub](https://github.com/google-gemini/gemini-live-api-examples)


In [1]:
"""
Project 16: Digital Product Funnel & Retention Analytics
Libraries: numpy, pandas, matplotlib only.

Run:  python funnel_retention_analytics.py
Output: ./funnel_outputs/  (PNG figures + CSV tables), plus a printed report.

Mapping to the required work
  1) Event grain + dedupe ............ clean_events()
  2) Ordering / identity / sessions .. stitch_identity(), sessionize(), validate_events()
  3) Funnel stages + conversion ...... ordered_funnel(), funnel_table()
  4) NumPy bootstrap + comparisons ... boot_counts(), boot_means(), compare_to_rest()
  5) Cohort retention + repeat use ... retention_curve(), repeat_engagement()
  6) Funnels by device/source/cohort . segment_funnels()
  7) Alternative windows/definitions . session_gap_sensitivity(), definition_sensitivity()
  8) Experiment recommendations ...... device_gap_adjusted(), recommendations()

Data: a documented synthetic event log (generate_events). To use a real dataset
(e.g. RetailRocket / Kaggle e-commerce events) see load_real_events() at the bottom.
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
OUT_DIR = "funnel_outputs"
STAGES = ["page_view", "product_view", "add_to_cart", "checkout_start", "purchase"]
SESSION_GAP_MIN = 30                      # default session timeout (minutes)
MAIN_WINDOW = pd.Timedelta(days=7)        # funnel must complete within 7 days of entry
N_BOOT = 1000

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


# ----------------------------------------------------------------------------
# 0. SYNTHETIC EVENT LOG (documented generator)
# ----------------------------------------------------------------------------
def generate_events(n_users=20000, n_days=120, start="2025-01-01", seed=SEED):
    """
    Documented synthetic event log. Columns:
      event_id     unique id of the emitted row (retries re-emit the SAME id)
      anon_id      cookie / device id (a user on 2 devices has 2 anon_ids)
      user_id      account id, only present after login (else None)
      event_name   one of STAGES
      ts           event timestamp
      device, source (source = utm source on the event; 'direct' on some return visits)
      true_user    ground-truth user (synthetic only; used to audit identity stitching)

    Built-in data problems to detect / fix:
      * 3% exact duplicate rows (same event_id)         -> retry/double-send
      * 2% near duplicates (new id, <1.5s later)        -> double-fired tags
      * 1% of late-funnel events with skewed timestamps -> ordering violations
      * 0.2% missing timestamps
      * login happens mid-journey -> anon events must be stitched to user_id
      * 15% of users use a 2nd device (a second anon_id)
    Built-in structure (so findings are non-trivial):
      * mobile has checkout friction; social traffic skews mobile (confounding!)
      * email has high intent and revisit rate
    """
    rng = np.random.default_rng(seed)
    base = pd.Timestamp(start)
    src_names = ["organic", "paid_search", "social", "email", "referral"]
    src_p = [.35, .25, .20, .10, .10]
    src_intent = {"organic": 1.0, "paid_search": .9, "social": .7, "email": 1.35, "referral": 1.1}
    src_revisit = {"organic": 1.0, "paid_search": .7, "social": .8, "email": 2.0, "referral": 1.2}
    dev_names = np.array(["mobile", "desktop", "tablet"])
    dev_given_src = {"organic": [.50, .43, .07], "paid_search": [.50, .43, .07],
                     "social": [.80, .15, .05], "email": [.55, .38, .07],
                     "referral": [.45, .48, .07]}
    dev_checkout = {"mobile": .78, "desktop": 1.1, "tablet": 1.0}

    src = rng.choice(src_names, n_users, p=src_p)
    dev = np.empty(n_users, dtype=object)
    for s in src_names:
        m = src == s
        dev[m] = rng.choice(dev_names, m.sum(), p=dev_given_src[s])
    first_day = rng.integers(0, n_days - 14, n_users)
    revisit = np.array([src_revisit[s] for s in src])
    n_sess = np.minimum(1 + rng.poisson(0.9 * revisit), 12)

    # ---- sessions
    uid = np.repeat(np.arange(n_users), n_sess)
    k = np.arange(len(uid)) - np.repeat(np.cumsum(n_sess) - n_sess, n_sess)
    gap_days = np.where(k == 0, 0.0, rng.exponential(6.0, len(uid)))
    cum = pd.Series(gap_days).groupby(uid).cumsum().to_numpy()
    start_sec = (first_day[uid] + cum) * 86400 + rng.integers(7 * 3600, 23 * 3600, len(uid))
    keep = start_sec < n_days * 86400
    uid, k, start_sec = uid[keep], k[keep], start_sec[keep]
    n = len(uid)

    has2 = rng.random(n_users) < .15
    switch = has2[uid] & (k >= 1) & (rng.random(n) < .5)
    alt_dev = np.where(dev == "mobile", "desktop", "mobile")
    s_dev = np.where(switch, alt_dev[uid], dev[uid])
    anon = np.array([f"a{u}_{1 + int(sw)}" for u, sw in zip(uid, switch)])
    s_src = np.where(k == 0, src[uid],
                     np.where(rng.random(n) < .5, "direct", src[uid]))

    # ---- funnel progression per session
    intent = np.array([src_intent[s] for s in src[uid]]) ** .6 * np.where(k > 0, 1.1, 1.0)
    dco = np.array([dev_checkout[d] for d in s_dev])
    p = [np.clip(.62 * intent, 0, .97), np.clip(.34 * intent, 0, .97),
         np.clip(.55 * intent * dco, 0, .97), np.clip(.60 * intent * dco, 0, .97)]
    reach = [np.ones(n, bool)]
    for j in range(4):
        reach.append(reach[-1] & (rng.random(n) < p[j]))
    t = [start_sec.astype(float)]
    t.append(t[0] + 10 + rng.exponential(60, n))
    t.append(t[1] + 15 + rng.exponential(120, n))
    t.append(t[2] + 20 + rng.exponential(300, n))
    t.append(t[3] + 30 + rng.exponential(600, n))       # long tail -> session-gap sensitivity

    # ---- login: ~60% of users log in at a random session; checkout/purchase require login
    login_k = np.where(rng.random(n_users) < .6, rng.integers(0, 3, n_users), 99)

    parts = []
    for j, name in enumerate(STAGES):
        m = reach[j]
        logged = (k[m] > login_k[uid[m]]) | ((k[m] == login_k[uid[m]]) & (j >= 1)) | (j >= 3)
        parts.append(pd.DataFrame({
            "anon_id": anon[m],
            "user_id": np.where(logged, np.char.add("u", uid[m].astype(str)), None),
            "true_user": uid[m],
            "event_name": name,
            "stage": j,
            "ts_sec": t[j][m],
            "device": s_dev[m],
            "source": s_src[m]}))
    ev = pd.concat(parts, ignore_index=True)
    ev["ts"] = base + pd.to_timedelta(ev.pop("ts_sec"), unit="s")
    ev["event_id"] = np.arange(len(ev))
    N = len(ev)

    # ---- inject data-quality problems
    skew = (ev.stage.to_numpy() >= 2) & (rng.random(N) < .01)
    ev.loc[skew, "ts"] -= pd.to_timedelta(rng.uniform(600, 7200, skew.sum()), unit="s")
    dup_exact = ev.sample(frac=.03, random_state=seed)
    near = ev.sample(frac=.02, random_state=seed + 1).copy()
    near["event_id"] = np.arange(N, N + len(near))
    near["ts"] += pd.to_timedelta(rng.uniform(.1, 1.5, len(near)), unit="s")
    ev = pd.concat([ev, dup_exact, near], ignore_index=True)
    ev.loc[rng.random(len(ev)) < .002, "ts"] = pd.NaT
    ev = ev.drop(columns="stage").sample(frac=1, random_state=seed).reset_index(drop=True)
    return ev[["event_id", "anon_id", "user_id", "event_name", "ts", "device", "source", "true_user"]]


# ----------------------------------------------------------------------------
# 1-2. CLEANING, IDENTITY, SESSIONS, VALIDATION
# ----------------------------------------------------------------------------
def stitch_identity(df):
    """person_id = user_id if present, else the user_id this anon_id was later seen with,
    else the anon_id itself. Returns (df, stats)."""
    pairs = df.dropna(subset=["user_id"])[["anon_id", "user_id"]].drop_duplicates()
    conflicts = int((pairs.groupby("anon_id")["user_id"].nunique() > 1).sum())
    amap = pairs.drop_duplicates("anon_id").set_index("anon_id")["user_id"]
    df = df.copy()
    df["person_id"] = df["user_id"].fillna(df["anon_id"].map(amap)).fillna(df["anon_id"])
    return df, {"anon_ids_linked_to_multiple_users": conflicts,
                "anon_ids_stitched_to_a_user": int(len(amap))}


def clean_events(raw, near_dup_sec=2.0):
    """
    EVENT GRAIN: one row = one (person_id, event_name, timestamp) occurrence.
    Dedupe in two passes:
      a) same event_id  -> retried delivery of the identical row
      b) same person + event_name within `near_dup_sec` seconds -> double-fired tag
    """
    log = {"raw_rows": len(raw)}
    df = raw.copy()
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    log["null_timestamp_rows_dropped"] = int(df.ts.isna().sum())
    df = df.dropna(subset=["ts"])
    n0 = len(df)
    df = df.drop_duplicates("event_id")
    log["exact_duplicates_removed"] = n0 - len(df)
    df, stitch_stats = stitch_identity(df)
    log.update(stitch_stats)
    df = df.sort_values(["person_id", "event_name", "ts"])
    gap = df.groupby(["person_id", "event_name"])["ts"].diff().dt.total_seconds()
    near = (gap < near_dup_sec).to_numpy()
    log["near_duplicates_removed"] = int(near.sum())
    df = df[~near].sort_values(["person_id", "ts"]).reset_index(drop=True)
    log["clean_rows"] = len(df)
    return df, log


def sessionize(df, gap_min=SESSION_GAP_MIN):
    """New session when the gap to the person's previous event exceeds gap_min minutes."""
    df = df.sort_values(["person_id", "ts"]).copy()
    gap = df.groupby("person_id")["ts"].diff()
    new = gap.isna() | (gap > pd.Timedelta(minutes=gap_min))
    df["session_id"] = new.cumsum().to_numpy()
    df["session_no"] = new.astype(int).groupby(df["person_id"]).cumsum().to_numpy()
    return df


def validate_events(df, log):
    """Identity, ordering and session sanity checks -> DataFrame of (check, value)."""
    rows = [(k, v) for k, v in log.items()]
    first = df[df.event_name.isin(STAGES)].groupby(["person_id", "event_name"])["ts"].min().unstack()
    rows.append(("persons", df.person_id.nunique()))
    rows.append(("sessions", df.session_id.nunique()))
    rows.append(("share_events_with_user_id", df.user_id.notna().mean()))
    if "true_user" in df:
        rows.append(("identity_fragmentation (persons per true user; 1.0 ideal)",
                     df.person_id.nunique() / df.true_user.nunique()))
        rows.append(("share_persons_mixing_>1_true_user (should be 0)",
                     (df.groupby("person_id")["true_user"].nunique() > 1).mean()))
    for a, b in zip(STAGES[:-1], STAGES[1:]):
        both = first[[a, b]].dropna()
        rows.append((f"order_violation {b} before {a}", (both[b] < both[a]).mean()))
    rows.append(("purchase_without_checkout_start_persons",
                 int((first["purchase"].notna() & first["checkout_start"].isna()).sum())))
    rows.append(("sessions_spanning_>1_device_share",
                 (df.groupby("session_id")["device"].nunique() > 1).mean()))
    dur = df.groupby("session_id")["ts"].agg(lambda s: (s.max() - s.min()).total_seconds() / 60)
    rows.append(("session_duration_min_median", dur.median()))
    rows.append(("session_duration_min_p95", dur.quantile(.95)))
    return pd.DataFrame(rows, columns=["check", "value"])


def person_attributes(df):
    """First-touch attributes per person + acquisition cohort."""
    d = df.sort_values("ts")
    a = d.groupby("person_id").agg(first_ts=("ts", "min"), device=("device", "first"))
    s = d[d.source != "direct"].groupby("person_id")["source"].first()
    a["source"] = s.reindex(a.index).fillna("direct")
    a["cohort_week"] = a.first_ts.dt.normalize() - pd.to_timedelta(a.first_ts.dt.weekday, unit="D")
    a["cohort_month"] = a.first_ts.dt.to_period("M").astype(str)
    return a


# ----------------------------------------------------------------------------
# 3. FUNNELS
# ----------------------------------------------------------------------------
def ordered_funnel(df, key, stages=STAGES, window=None, ordered=True):
    """
    Boolean matrix (rows = key, cols = stages): did the unit reach each stage?
      key     ['person_id'] (user-level) or ['session_id'] (session-level)
      ordered stage k must occur at/after the time stage k-1 was reached
      window  stage k must occur within `window` of the entry (first page_view)
    The funnel is nested: reaching stage k requires reaching k-1.
    """
    ev = df[df.event_name.isin(stages)][key + ["event_name", "ts"]]
    entry = ev[ev.event_name == stages[0]].groupby(key)["ts"].min()
    entry.name = "t0"
    cols, prev = {stages[0]: entry}, entry
    entry_df = entry.reset_index()
    for s in stages[1:]:
        e = (ev[ev.event_name == s]
             .merge(prev.rename("prev").reset_index(), on=key)
             .merge(entry_df, on=key))
        if ordered:
            e = e[e.ts >= e.prev]
        if window is not None:
            e = e[e.ts <= e.t0 + window]
        t = e.groupby(key)["ts"].min()
        cols[s] = t
        prev = t
    return pd.DataFrame(cols).notna()


def funnel_counts(reached):
    c = reached.sum()
    return pd.DataFrame({"users": c, "step_conv": c / c.shift(1), "overall_conv": c / c.iloc[0]})


# ----------------------------------------------------------------------------
# 4. NUMPY BOOTSTRAP
# ----------------------------------------------------------------------------
def boot_counts(arr, n_boot, rng):
    """Resample units with replacement; return (n_boot, n_stages) reached-counts."""
    n, k = arr.shape
    out = np.empty((n_boot, k))
    chunk = max(1, int(2_000_000 // n))
    for i in range(0, n_boot, chunk):
        m = min(chunk, n_boot - i)
        idx = rng.integers(0, n, size=(m, n))
        out[i:i + m] = arr[idx].sum(axis=1)
    return out


def boot_means(y, n_boot, rng):
    n = len(y)
    out = np.empty(n_boot)
    chunk = max(1, int(2_000_000 // n))
    for i in range(0, n_boot, chunk):
        m = min(chunk, n_boot - i)
        out[i:i + m] = y[rng.integers(0, n, size=(m, n))].mean(axis=1)
    return out


def funnel_table(reached, n_boot=N_BOOT, rng=None):
    """Funnel counts with 95% bootstrap CIs for step and overall conversion."""
    rng = rng or np.random.default_rng(SEED)
    tab = funnel_counts(reached)
    c = boot_counts(reached.to_numpy(), n_boot, rng)
    overall = c / c[:, [0]]
    step = np.full_like(c, np.nan)
    step[:, 1:] = c[:, 1:] / np.maximum(c[:, :-1], 1)
    tab["overall_lo"], tab["overall_hi"] = np.percentile(overall, [2.5, 97.5], axis=0)
    tab["step_lo"], tab["step_hi"] = np.nan, np.nan
    tab.iloc[1:, tab.columns.get_loc("step_lo")] = np.percentile(step[:, 1:], 2.5, axis=0)
    tab.iloc[1:, tab.columns.get_loc("step_hi")] = np.percentile(step[:, 1:], 97.5, axis=0)
    return tab


def compare_to_rest(y, mask, n_boot, rng):
    """Difference in conversion (segment - everyone else), bootstrap 95% CI and p-value."""
    a, b = y[mask], y[~mask]
    d = boot_means(a, n_boot, rng) - boot_means(b, n_boot, rng)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return a.mean() - b.mean(), *np.percentile(d, [2.5, 97.5]), max(p, 1 / n_boot)


# ----------------------------------------------------------------------------
# 5. COHORT RETENTION + REPEAT ENGAGEMENT
# ----------------------------------------------------------------------------
def weekly_activity(df, attrs):
    """Unique (person, weeks-since-first-seen) activity rows."""
    d = df[["person_id", "ts"]].join(attrs["first_ts"], on="person_id")
    d["k"] = (d.ts - d.first_ts).dt.days // 7
    return d.drop_duplicates(["person_id", "k"])[["person_id", "k"]]


def retention_curve(act, attrs, last_ts, col=None, K=8, min_n=50):
    """Share of persons active in week k after first seen, counting only persons whose
    week k is fully observed (handles right-censoring). Returns DataFrame (k x group)."""
    groups = [("all", attrs)] if col is None else [(g, a) for g, a in attrs.groupby(col)]
    out = {}
    for name, a in groups:
        r = []
        for k in range(K + 1):
            obs = a.index[(a.first_ts + pd.Timedelta(days=7 * (k + 1))) <= last_ts]
            if len(obs) < min_n:
                r.append(np.nan)
                continue
            active = act.loc[(act.k == k) & act.person_id.isin(obs), "person_id"].nunique()
            r.append(active / len(obs))
        out[name] = r
    return pd.DataFrame(out, index=range(K + 1))


def repeat_engagement(df, attrs, by="source"):
    d = df.assign(is_p=df.event_name.eq("purchase"))
    p = d.groupby("person_id").agg(sessions=("session_id", "nunique"), purchases=("is_p", "sum"))
    st = d.groupby(["person_id", "session_id"])["ts"].min().reset_index().sort_values(["person_id", "ts"])
    st["n"] = st.groupby("person_id").cumcount()
    s1 = st[st.n == 0].set_index("person_id")["ts"]
    s2 = st[st.n == 1].set_index("person_id")["ts"]
    p["days_to_2nd_session"] = (s2 - s1).dt.total_seconds() / 86400
    p = p.join(attrs[[by]])

    def summ(g):
        buyers = g[g.purchases >= 1]
        return pd.Series({
            "persons": len(g),
            "mean_sessions": g.sessions.mean(),
            "pct_2plus_sessions": (g.sessions >= 2).mean(),
            "median_days_to_2nd_session": g.days_to_2nd_session.median(),
            "pct_buyers": len(buyers) / len(g),
            "pct_repeat_buyers_of_buyers": (buyers.purchases >= 2).mean() if len(buyers) else np.nan})
    tab = pd.concat([summ(p).rename("ALL")] + [summ(g).rename(n) for n, g in p.groupby(by)], axis=1).T
    return tab, p


# ----------------------------------------------------------------------------
# 6. SEGMENT FUNNELS
# ----------------------------------------------------------------------------
def segment_funnels(reached, attrs, cols=("device", "source", "cohort_month"), n_boot=N_BOOT, rng=None):
    rng = rng or np.random.default_rng(SEED)
    a = attrs.loc[reached.index]
    arr = reached.to_numpy()
    y = reached[STAGES[-1]].to_numpy().astype(float)
    rows, comps = [], []
    for col in cols:
        for lvl in sorted(a[col].unique()):
            m = (a[col] == lvl).to_numpy()
            if m.sum() < 200:
                continue
            c = arr[m].sum(axis=0)
            ci = np.percentile(boot_means(y[m], n_boot, rng), [2.5, 97.5])
            rows.append({"segment": col, "level": lvl, "persons": int(m.sum()),
                         **{f"{s}": c[i] / c[0] for i, s in enumerate(STAGES)},
                         **{f"step_{s}": (c[i] / c[i - 1] if i else np.nan) for i, s in enumerate(STAGES)},
                         "purchase_lo": ci[0], "purchase_hi": ci[1]})
            d, lo, hi, p = compare_to_rest(y, m, n_boot, rng)
            comps.append({"segment": col, "level": lvl, "purchase_conv": y[m].mean(),
                          "diff_vs_rest": d, "diff_lo": lo, "diff_hi": hi, "p_boot": p})
    return pd.DataFrame(rows), pd.DataFrame(comps)


def device_gap_adjusted(y, dev, src, a="mobile", b="desktop", n_boot=500, rng=None):
    """Raw vs source-standardised conversion gap between two devices (observational
    confounding check: social traffic skews mobile). Bootstrap CIs."""
    rng = rng or np.random.default_rng(SEED)
    dev = np.asarray(dev)
    keep = (dev == a) | (dev == b)
    y, dev = y[keep].astype(float), dev[keep]
    sc, levels = pd.factorize(np.asarray(src)[keep])
    S = len(levels)

    def stat(y, dev, sc):
        w = np.bincount(sc, minlength=S) / len(sc)
        rates = []
        for lvl in (a, b):
            m = dev == lvl
            num = np.bincount(sc[m], weights=y[m], minlength=S)
            den = np.bincount(sc[m], minlength=S)
            rates.append(np.divide(num, den, out=np.zeros(S), where=den > 0))
        return y[dev == a].mean() - y[dev == b].mean(), ((rates[0] - rates[1]) * w).sum()

    raw, adj = stat(y, dev, sc)
    boots = np.array([stat(y[i], dev[i], sc[i]) for i in
                      (rng.integers(0, len(y), len(y)) for _ in range(n_boot))])
    r_ci, a_ci = np.percentile(boots[:, 0], [2.5, 97.5]), np.percentile(boots[:, 1], [2.5, 97.5])
    tbl = (pd.DataFrame({"y": y, "device": dev, "source": np.asarray(src)[keep]})
           .pivot_table(index="source", columns="device", values="y", aggfunc="mean"))
    return {"raw_gap": raw, "raw_ci": r_ci, "adj_gap": adj, "adj_ci": a_ci, "table": tbl}


# ----------------------------------------------------------------------------
# 7. SENSITIVITY: SESSION WINDOWS + FUNNEL DEFINITIONS
# ----------------------------------------------------------------------------
def session_gap_sensitivity(df, eligible, gaps=(10, 30, 60, 120)):
    rows = []
    for g in gaps:
        s = sessionize(df, g)
        s = s[s.person_id.isin(eligible)]
        r = ordered_funnel(s, ["session_id"])
        rows.append({"gap_min": g, "sessions": s.session_id.nunique(),
                     "sessions_per_person": s.session_id.nunique() / s.person_id.nunique(),
                     "session_pv_to_purchase": r["purchase"].mean(),
                     "session_pv_to_checkout": r["checkout_start"].mean()})
    return pd.DataFrame(rows)


def definition_sensitivity(df_sess, eligible, n_boot=500, rng=None):
    rng = rng or np.random.default_rng(SEED)
    d = df_sess[df_sess.person_id.isin(eligible)]
    defs = [
        ("User | strict order | 7d window", ["person_id"], dict(window=pd.Timedelta(days=7))),
        ("User | strict order | 24h window", ["person_id"], dict(window=pd.Timedelta(days=1))),
        ("User | strict order | no window", ["person_id"], dict()),
        ("User | any order | 7d window", ["person_id"], dict(window=pd.Timedelta(days=7), ordered=False)),
        (f"Session ({SESSION_GAP_MIN}m) | strict order", ["session_id"], dict()),
    ]
    rows = []
    for name, key, kw in defs:
        r = ordered_funnel(d, key, **kw)
        y = r["purchase"].to_numpy().astype(float)
        lo, hi = np.percentile(boot_means(y, n_boot, rng), [2.5, 97.5])
        rows.append({"definition": name, "entrants": len(r), "purchase_conv": y.mean(),
                     "lo": lo, "hi": hi, "cart_to_purchase": r["purchase"].sum() / max(r["add_to_cart"].sum(), 1)})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# 8. RECOMMENDATIONS
# ----------------------------------------------------------------------------
def n_per_arm(p, rel_lift, alpha_z=1.96, power_z=0.84):
    """Two-proportion sample size per arm (normal approximation, 5% two-sided, 80% power)."""
    return int(np.ceil(2 * (alpha_z + power_z) ** 2 * p * (1 - p) / (p * rel_lift) ** 2))


def recommendations(ftab, seg, comps, gap, rep):
    step = ftab["step_conv"].iloc[1:]
    weakest = step.idxmin()
    mob = seg[(seg.segment == "device") & (seg.level == "mobile")].iloc[0]
    desk = seg[(seg.segment == "device") & (seg.level == "desktop")].iloc[0]
    soc = seg[(seg.segment == "source") & (seg.level == "social")].iloc[0]
    eml = seg[(seg.segment == "source") & (seg.level == "email")].iloc[0]
    print("\n" + "=" * 78 + "\nFINDINGS (observational)\n" + "=" * 78)
    print(f"- Weakest funnel step: {weakest} (step conversion {step.min():.1%}).")
    print(f"- Overall pageview->purchase (7d): {ftab.overall_conv.iloc[-1]:.2%} "
          f"[{ftab.overall_lo.iloc[-1]:.2%}, {ftab.overall_hi.iloc[-1]:.2%}].")
    print(f"- Mobile vs desktop checkout->purchase: {mob.step_purchase:.1%} vs {desk.step_purchase:.1%}.")
    print(f"- Raw mobile-desktop purchase gap {gap['raw_gap']:+.2%} "
          f"[{gap['raw_ci'][0]:+.2%}, {gap['raw_ci'][1]:+.2%}]; after standardising for source mix "
          f"{gap['adj_gap']:+.2%} [{gap['adj_ci'][0]:+.2%}, {gap['adj_ci'][1]:+.2%}].")
    print(f"- Social purchase conv {soc.purchase:.2%} vs email {eml.purchase:.2%}; "
          f"email users are also the most likely to return (see repeat table).")
    print("- These are ASSOCIATIONS: device/source/cohort are not randomly assigned, users self-select")
    print("  into devices and channels, and retention differences by first-session behaviour reflect")
    print("  selection (engaged users do more of everything), not the effect of any feature.")

    print("\n" + "=" * 78 + "\nEXPERIMENTS TO RUN (randomised -> causal)\n" + "=" * 78)
    lift = 0.10
    exps = [
        ("Mobile checkout simplification (wallet pay / fewer fields)",
         "Mobile users reaching add_to_cart", "cart->purchase", mob.step_checkout_start * mob.step_purchase,
         "Randomise at user_id level; guardrails: refund rate, payment errors, load time."),
        (f"Fix weakest step: improve '{weakest}' experience",
         "Users reaching the preceding stage", weakest, float(step.min()),
         "Randomise at user level (not session) to avoid contamination across visits."),
        ("Social landing-page personalisation",
         "New social-acquired users", "pageview->purchase", soc.purchase,
         "Randomise on first visit; analyse by intent-to-treat; segment pre-registered."),
        ("Email re-engagement cadence (vs holdout)",
         "Users with 1 session", "has 2+ sessions (return visit)",
         float(rep.loc["ALL", "pct_2plus_sessions"]),
         "Hold out 10% with no email; measure retention AND unsubscribes."),
    ]
    rows = [{"experiment": n, "population": pop, "primary_metric": m, "baseline": b,
             f"n_per_arm_for_{lift:.0%}_rel_lift": n_per_arm(b, lift), "design_note": note}
            for n, pop, m, b, note in exps]
    ex = pd.DataFrame(rows)
    for r in rows:
        print(f"\n* {r['experiment']}\n  metric: {r['primary_metric']} | baseline {r['baseline']:.1%} | "
              f"n/arm ≈ {r[f'n_per_arm_for_{lift:.0%}_rel_lift']:,}\n  {r['design_note']}")
    print("\nCausal checklist: pre-register metric + MDE, randomise on a stable id (person_id), run")
    print("full weekly cycles, check sample-ratio mismatch, report CIs, do not peek-and-stop.")
    return ex


# ----------------------------------------------------------------------------
# PLOTTING
# ----------------------------------------------------------------------------
def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, name), dpi=140)
    plt.close(fig)


def plot_funnel(ftab):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    y = np.arange(len(ftab))[::-1]
    ax.barh(y, ftab.overall_conv, color="#3b6fb6", alpha=.85)
    ax.errorbar(ftab.overall_conv, y, xerr=[ftab.overall_conv - ftab.overall_lo,
                ftab.overall_hi - ftab.overall_conv], fmt="none", ecolor="k", capsize=3)
    for yi, (s, r) in zip(y, ftab.iterrows()):
        txt = f"{r.overall_conv:.1%}" + ("" if np.isnan(r.step_conv) else f"  (step {r.step_conv:.1%})")
        ax.text(r.overall_hi + .01, yi, txt, va="center", fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(ftab.index)
    ax.set_xlim(0, 1.25)
    ax.set_xlabel("share of entrants (95% bootstrap CI)")
    ax.set_title("Overall funnel (user-level, strict order, 7-day window)")
    savefig(fig, "01_funnel.png")


def plot_segments(seg, cols=("device", "source", "cohort_month")):
    fig, axes = plt.subplots(2, len(cols), figsize=(5 * len(cols), 8))
    for j, col in enumerate(cols):
        s = seg[seg.segment == col]
        for _, r in s.iterrows():
            axes[0, j].plot(STAGES, [r[x] for x in STAGES], marker="o", label=r.level)
        axes[0, j].set_title(f"Funnel by {col}")
        axes[0, j].tick_params(axis="x", rotation=30)
        axes[0, j].set_ylabel("share of entrants")
        axes[0, j].legend(fontsize=8)
        x = np.arange(len(s))
        axes[1, j].bar(x, s.purchase, color="#3b6fb6", alpha=.85)
        axes[1, j].errorbar(x, s.purchase, yerr=[s.purchase - s.purchase_lo, s.purchase_hi - s.purchase],
                            fmt="none", ecolor="k", capsize=3)
        axes[1, j].set_xticks(x)
        axes[1, j].set_xticklabels(s.level, rotation=30)
        axes[1, j].set_title(f"Purchase conversion by {col} (95% CI)")
    savefig(fig, "02_segments.png")


def plot_retention(heat, curve_all, curve_src, rep_p):
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
    ax = axes[0]
    im = ax.imshow(heat.values, aspect="auto", cmap="Blues", vmin=0, vmax=max(np.nanmax(heat.values), .01))
    ax.set_xticks(range(heat.shape[1]))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(range(heat.shape[0]))
    ax.set_yticklabels([d.strftime("%Y-%m-%d") for d in heat.index], fontsize=7)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=6,
                        color="white" if v > .5 * np.nanmax(heat.values) else "black")
    ax.set_xlabel("weeks since first seen")
    ax.set_ylabel("acquisition cohort (week)")
    ax.set_title("Weekly cohort retention (blank = not yet observable)")
    fig.colorbar(im, ax=ax, fraction=.04)
    ax = axes[1]
    ax.plot(curve_all.index, curve_all["all"], "k-o", lw=2, label="all")
    for c in curve_src.columns:
        ax.plot(curve_src.index, curve_src[c], marker=".", label=c)
    ax.set_xlabel("weeks since first seen")
    ax.set_ylabel("share active")
    ax.set_title("Retention curve by acquisition source")
    ax.legend(fontsize=8)
    ax = axes[2]
    vc = rep_p.sessions.clip(upper=8).value_counts(normalize=True).sort_index()
    ax.bar(vc.index, vc.values, color="#3b6fb6")
    ax.set_xticks(vc.index)
    ax.set_xticklabels([str(i) if i < 8 else "8+" for i in vc.index])
    ax.set_xlabel("sessions per person")
    ax.set_ylabel("share of persons")
    ax.set_title("Repeat engagement")
    savefig(fig, "03_retention.png")


def plot_sensitivity(gap_tab, def_tab):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    ax = axes[0]
    ax.plot(gap_tab.gap_min, gap_tab.session_pv_to_purchase, "o-", label="session pageview→purchase")
    ax.set_xlabel("session timeout (min)")
    ax.set_ylabel("session-level conversion")
    ax2 = ax.twinx()
    ax2.plot(gap_tab.gap_min, gap_tab.sessions_per_person, "s--", color="tab:red", label="sessions / person")
    ax2.set_ylabel("sessions per person")
    ax.set_title("Sensitivity to session window")
    ax.legend(loc="lower left", fontsize=8)
    ax2.legend(loc="upper right", fontsize=8)
    ax = axes[1]
    y = np.arange(len(def_tab))[::-1]
    ax.errorbar(def_tab.purchase_conv, y, xerr=[def_tab.purchase_conv - def_tab.lo, def_tab.hi - def_tab.purchase_conv],
                fmt="o", capsize=4)
    ax.set_yticks(y)
    ax.set_yticklabels(def_tab.definition, fontsize=8)
    ax.set_xlabel("pageview→purchase conversion (95% CI)")
    ax.set_title("Sensitivity to funnel definition")
    savefig(fig, "04_sensitivity.png")


def plot_device_confounding(gap):
    t = gap["table"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    t.plot(kind="bar", ax=axes[0], color=["#d9822b", "#3b6fb6"])
    axes[0].set_ylabel("purchase conversion")
    axes[0].set_title("Conversion within each source (mobile vs desktop)")
    axes[0].tick_params(axis="x", rotation=30)
    ax = axes[1]
    vals = [gap["raw_gap"], gap["adj_gap"]]
    cis = [gap["raw_ci"], gap["adj_ci"]]
    ax.errorbar(vals, [1, 0], xerr=[[v - c[0] for v, c in zip(vals, cis)], [c[1] - v for v, c in zip(vals, cis)]],
                fmt="o", capsize=5)
    ax.axvline(0, color="grey", ls="--")
    ax.set_yticks([1, 0])
    ax.set_yticklabels(["raw gap", "source-standardised gap"])
    ax.set_xlabel("mobile − desktop purchase conversion")
    ax.set_title("Does channel mix explain the device gap?")
    savefig(fig, "05_device_confounding.png")


# ----------------------------------------------------------------------------
# REAL DATA HOOK
# ----------------------------------------------------------------------------
def load_real_events(path, colmap, event_map):
    """
    Map a real event CSV to the schema used above, e.g. RetailRocket:
      colmap   = {"timestamp": "ts", "visitorid": "anon_id", "event": "event_name"}
      event_map= {"view": "product_view", "addtocart": "add_to_cart", "transaction": "purchase"}
    then call main(raw=...) with STAGES edited to the stages that exist
    (no checkout_start there -> STAGES = ["product_view","add_to_cart","purchase"]).
    Missing columns (device, source, user_id) are filled with 'unknown' / None.
    """
    d = pd.read_csv(path).rename(columns=colmap)
    d["event_name"] = d["event_name"].map(event_map)
    d = d.dropna(subset=["event_name"])
    if np.issubdtype(d["ts"].dtype, np.number):
        d["ts"] = pd.to_datetime(d["ts"], unit="ms")
    for c, v in [("device", "unknown"), ("source", "unknown"), ("user_id", None)]:
        if c not in d:
            d[c] = v
    d["event_id"] = np.arange(len(d))
    return d[["event_id", "anon_id", "user_id", "event_name", "ts", "device", "source"]]


# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------
def main(raw=None):
    os.makedirs(OUT_DIR, exist_ok=True)
    rng = np.random.default_rng(SEED)
    raw = generate_events() if raw is None else raw

    print("=" * 78 + "\n1-2. CLEANING, IDENTITY, SESSIONS, VALIDATION\n" + "=" * 78)
    df, log = clean_events(raw)
    df = sessionize(df, SESSION_GAP_MIN)
    val = validate_events(df, log)
    print(val.to_string(index=False))
    val.to_csv(f"{OUT_DIR}/validation_checks.csv", index=False)

    attrs = person_attributes(df)
    last_ts = df.ts.max()
    eligible = attrs.index[attrs.first_ts <= last_ts - MAIN_WINDOW]   # full 7-day follow-up
    d_main = df[df.person_id.isin(eligible)]

    print("\n" + "=" * 78 + "\n3-4. FUNNEL + BOOTSTRAP CIs\n" + "=" * 78)
    reached = ordered_funnel(d_main, ["person_id"], window=MAIN_WINDOW)
    ftab = funnel_table(reached, N_BOOT, rng)
    print(ftab.to_string())
    ftab.to_csv(f"{OUT_DIR}/funnel_overall.csv")
    plot_funnel(ftab)

    print("\n" + "=" * 78 + "\n6. FUNNELS BY DEVICE / SOURCE / COHORT MONTH\n" + "=" * 78)
    seg, comps = segment_funnels(reached, attrs, rng=rng)
    print(seg.round(4).to_string(index=False))
    print("\nSegment vs rest (purchase conversion, bootstrap 95% CI):")
    print(comps.round(4).to_string(index=False))
    seg.to_csv(f"{OUT_DIR}/funnel_by_segment.csv", index=False)
    comps.to_csv(f"{OUT_DIR}/segment_comparisons.csv", index=False)
    plot_segments(seg)

    a = attrs.loc[reached.index]
    gap = device_gap_adjusted(reached["purchase"].to_numpy(), a.device.to_numpy(), a.source.to_numpy(), rng=rng)
    plot_device_confounding(gap)

    print("\n" + "=" * 78 + "\n5. COHORT RETENTION + REPEAT ENGAGEMENT\n" + "=" * 78)
    act = weekly_activity(df, attrs)
    heat = retention_curve(act, attrs, last_ts, col="cohort_week", K=10).T.dropna(how="all")
    curve_all = retention_curve(act, attrs, last_ts, K=8)
    curve_src = retention_curve(act, attrs, last_ts, col="source", K=8)
    print("Weekly retention curve (all users):")
    print(curve_all.T.round(3).to_string())
    rep, rep_p = repeat_engagement(df, attrs, by="source")
    print("\nRepeat engagement by acquisition source:")
    print(rep.round(3).to_string())
    heat.to_csv(f"{OUT_DIR}/cohort_retention_matrix.csv")
    rep.to_csv(f"{OUT_DIR}/repeat_engagement.csv")
    plot_retention(heat, curve_all, curve_src, rep_p)

    print("\n" + "=" * 78 + "\n7. SENSITIVITY: SESSION WINDOWS & FUNNEL DEFINITIONS\n" + "=" * 78)
    gap_tab = session_gap_sensitivity(df, eligible)
    def_tab = definition_sensitivity(df, eligible, rng=rng)
    print(gap_tab.to_string(index=False))
    print()
    print(def_tab.to_string(index=False))
    gap_tab.to_csv(f"{OUT_DIR}/sensitivity_session_gap.csv", index=False)
    def_tab.to_csv(f"{OUT_DIR}/sensitivity_definitions.csv", index=False)
    plot_sensitivity(gap_tab, def_tab)

    print("\n" + "=" * 78 + "\n8. RECOMMENDATIONS\n" + "=" * 78)
    ex = recommendations(ftab, seg, comps, gap, rep)
    ex.to_csv(f"{OUT_DIR}/experiment_plan.csv", index=False)
    print(f"\nSaved figures + tables to ./{OUT_DIR}/")


if __name__ == "__main__":
    main()

1-2. CLEANING, IDENTITY, SESSIONS, VALIDATION
                                                    check       value
                                                 raw_rows 82,696.0000
                              null_timestamp_rows_dropped    170.0000
                                 exact_duplicates_removed  2,349.0000
                        anon_ids_linked_to_multiple_users      0.0000
                              anon_ids_stitched_to_a_user  8,544.0000
                                  near_duplicates_removed  1,567.0000
                                               clean_rows 78,610.0000
                                                  persons 20,743.0000
                                                 sessions 38,041.0000
                                share_events_with_user_id      0.3103
identity_fragmentation (persons per true user; 1.0 ideal)      1.0375
          share_persons_mixing_>1_true_user (should be 0)      0.0000
            order_violation product_view bef